In [2]:
import gurobipy as gp
from gurobipy import GRB
from utils.data import *
import math
import warnings
import plotly.express as px
import tqdm
import datetime as dt
from scipy.stats import norm
from gurobipy import GRB, nlfunc

def sigfig(x, n):
    y = f"{{:.{len(str(x))}f}}".format(x).replace(".", "")

    for i in range(1, len(y)+1):
        if y[:i] == "0"*i: continue
        else: 
            leading_zeros = i - 1
            break

    slice_ = min(len(y)+1, leading_zeros+n+1)

    return f"{{:.{len(str(x))}f}}".format(x)[:slice_]

start_str = dt.datetime.now().strftime("%Y%m%d%H%M%S")

In [3]:
start = dt.datetime(year=2000, month=1, day=1)
end = dt.datetime(year=2023, month=1, day=1)

all_stocks = ["NVDA", "AAPL", 'GOOG', "AVGO", "WMT", "JPM", "LLY", "XOM", "KO", "LMT", "F"]

# CHOOSE OBJECTIVE:
# -`min_var`    : Minimize expected variance
# -`max_mu`     : Maximize expected returns
# -`max_sharpe` : Maximize expected ratio of returns to variance

mode = "max_mu"

n = len(all_stocks)

# COVARIATES (Inflation, risk-free rate)
#a = (0.005, 0.25)  # 2% yearly inflation, 0.25% risk free rate (represents circa 2015 "the good times")
#a = (0.022, 4.5)  # 9% yearly inflation, 4.5% risk free rate (represents post-covid inflation period, start of 2022)
a = (0.00742, 5) # 3% yearly inflation, 5% risk free rate (represents start of 2023 conditions)
#a = (0.005, 3)  # 2% yearly inflation, 3% risk free rate (represents ideal federal reserve conditions)


T = 1/4
r = (1+a[1]/100)**(1/4)-1
S = [float(yf.Ticker(s).history(period="1d").Open.iloc[0]) for s in all_stocks]

In [4]:
pce = load_pce()
effr = load_ffr()


mu = generate_mu(all_stocks, start, end)
Sigma = generate_Sigma(all_stocks, start, end)

new_sigma, sep_Sigma = add_covariates_to_covar(Sigma, all_stocks, [pce, effr], start, end)
new_mu, sep_mu = add_covariates_to_mu(mu, [pce, effr])

mu, Sigma = conditional_moments(sep_mu, sep_Sigma, a)

sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]


### Model with fixed $w$

In [5]:
def solve_delta_given_w(Delta0, w):
    m = gp.Model()
    m.setParam("OutputFlag", 0)
    m.setParam("MIPGap", gap)
    m.setParam("TimeLimit", time_limit)
    #m.setParam("Presolve", 0)
    
    sigma = [np.sqrt(Sigma[i, i]) for i in range(n)]

    mu_d = m.addMVar(n, lb=-GRB.INFINITY, name="expectation")
    Delta = m.addMVar(n, lb=0.1, ub=0.5, name="Delta")
    P = m.addMVar(n, lb=-GRB.INFINITY, name="put_price")

    x_pts_CDF = np.linspace(-5, 5, 101)
    y_pts_CDF = -np.array([math.erf(x/np.sqrt(2))*1/2+1/2 for x in x_pts_CDF])

    for i, D in enumerate(Delta0):
        Delta[i].Start = D

    Sigma_d = np.zeros((n, n), dtype=gp.Var)

    # COMPUTING PRICE OF PUT OPTION & TRUNCATED EXPECTATION
    for i in range(n):
        alpha_i = m.addVar(lb=-GRB.INFINITY)
        m.addConstr(alpha_i == -1/1.7*gp.nlfunc.log(1/Delta[i]-1))
        
        K_expr = (sigma[i]*alpha_i + mu[i] + 1)*S[i]

        d1 = m.addVar(lb=-GRB.INFINITY, name=f"d1_{i}")
        d2 = m.addVar(lb=-GRB.INFINITY, name=f"d2_{i}")

        m.addConstr(d1 == (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T)))
        m.addConstr(d2 == d1 - sigma[i]*np.sqrt(T))

        cdf_d1_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d1_{i}")
        cdf_d2_expr = m.addVar(lb=-GRB.INFINITY, name=f"cdf_d2_{i}")

        cdf_d1 = m.addGenConstrPWL(d1, cdf_d1_expr, x_pts_CDF,\
                                                    y_pts_CDF)
        cdf_d2 = m.addGenConstrPWL(d2, cdf_d2_expr, x_pts_CDF,\
                                                    y_pts_CDF)
        
        phi_ai = m.addVar()
        m.addConstr(phi_ai == 1/np.sqrt(2*np.pi)*gp.nlfunc.exp(-1/2*alpha_i**2))

        P_expr = K_expr*np.exp(-r*T)*cdf_d2_expr - S[i]*cdf_d1_expr

        m.addConstr(P[i] == P_expr)
        m.addConstr(mu_d[i] == mu[i] + sigma[i]*(Delta[i]*alpha_i + phi_ai))
        
        var_i = m.addVar(lb=0, name=f"var_{i}")
        m.addConstr(
            var_i == sigma[i]**2 * (
                (1-Delta[i]) * (1 + alpha_i**2 * Delta[i])
            + alpha_i * phi_ai * (1 - 2*Delta[i])
            - phi_ai * phi_ai
            ),
            name=f"var_def_{i}"
        )
        Sigma_d[i, i] = var_i

        for j in range(i+1, n):
            rho_ij = Sigma[i, j] / np.sqrt(Sigma[i, i] * Sigma[j, j])

            covar_ij = m.addVar(lb=-GRB.INFINITY, name=f"covar_{i}_{j}")
            m.addConstr(
                covar_ij == sigma[i] * sigma[j] * rho_ij*(1-Delta[i])*(1-Delta[j])
                , name=f"covar_def_{i}_{j}"
            )
            
            Sigma_d[i, j] = covar_ij
            Sigma_d[j, i] = covar_ij

    # CONSTRAINTS TO COMPUTE RETURN/RISK RATIO EFFICIENTLY
    t = m.addVar(lb=-GRB.INFINITY)
    var_norm = m.addVar()
    var_norm_sq = m.addVar()

    m.addConstr(var_norm_sq == gp.quicksum([Sigma_d[i, i] * w[i]**2 for i in range(n)]) + \
                            2*gp.quicksum([Sigma_d[i, j] * w[i]*w[j] for i in range(n) for j in range(n) if j > i]))
    m.addConstr(var_norm_sq == var_norm**2)

    m.addConstr(gp.quicksum([(mu_d[i]-P[i]/S[i])*w[i] for i in range(n)]) -r >= var_norm * t)

    if mode == "max_mu":   
        m.setObjective(gp.quicksum(mu_d[i]*w[i] for i in range(n)), GRB.MAXIMIZE)
    elif mode == "max_sharpe":
        m.setObjective(t, GRB.MAXIMIZE)
    elif mode == "min_var":
        #m.addConstr(gp.quicksum([mu_d[i]*w[i] for i in range(n)]) >=0)
        m.setObjective(var_norm, GRB.MINIMIZE)
    else:
        print("CHOOSE VALID MODE")
        raise ValueError

    m.optimize()
    var = m.getVars()
    Deltas = np.array([float(v.X) for v in var if "Delta" in v.VarName])
    return m, m.ObjVal, Deltas

## Model with fixed $\Delta$

In [6]:
def pwl_cdf(x, x_pts, y_pts):
    """Replicate exactly what Gurobi's PWL does, in numpy."""
    return np.interp(x, x_pts, y_pts)

def solve_w_given_delta(Delta, w0, solve_mode):
    m = gp.Model()
    
    m.setParam("OutputFlag", 0)
    m.setParam("MIPGap", gap)
    m.setParam("TimeLimit", time_limit)
    #m.setParam("Presolve", 0)

    mu_d = np.zeros(n)
    P = np.zeros(n)

    x_pts_CDF = np.linspace(-5, 5, 101)
    y_pts_CDF = -np.array([math.erf(x/np.sqrt(2))*1/2+1/2 for x in x_pts_CDF])

    Sigma_d = np.zeros((n, n))

        # COMPUTING PRICE OF PUT OPTION & TRUNCATED EXPECTATION
    for i in range(n):
        alpha_i = -1/1.7*np.log(1/Delta[i]-1)
        
        K_expr = (sigma[i]*alpha_i + mu[i] + 1)*S[i]

        d1 = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
        d2 = d1 - sigma[i]*np.sqrt(T)

        cdf_d1 = pwl_cdf(d1, x_pts_CDF, y_pts_CDF)
        cdf_d2 = pwl_cdf(d2, x_pts_CDF, y_pts_CDF)
        
        phi_ai = 1/np.sqrt(2*np.pi)*np.exp(-1/2*alpha_i**2)

        P[i] = K_expr*np.exp(-r*T)*cdf_d2 - S[i]*cdf_d1

        mu_d[i] = mu[i] + sigma[i]*(Delta[i]*alpha_i + phi_ai)
        
        var_i = sigma[i]**2 * (
                (1-Delta[i]) * (1 + alpha_i**2 * Delta[i])
            + alpha_i * phi_ai * (1 - 2*Delta[i])
            - phi_ai * phi_ai)

        Sigma_d[i, i] = var_i

        for j in range(i+1, n):
            rho_ij = Sigma[i, j] / np.sqrt(Sigma[i, i] * Sigma[j, j])

            covar_ij = sigma[i] * sigma[j] * rho_ij*(1-Delta[i])*(1-Delta[j])
            
            Sigma_d[i, j] = covar_ij
            Sigma_d[j, i] = covar_ij

    # Cholesky decomposition — valid since Sigma_d is PD (proven in your paper)
    L = np.linalg.cholesky(Sigma_d)   # L @ L.T == Sigma_d

    w = m.addMVar(n, lb=0, ub=1, name="weight")
    for i, ws in enumerate(w0):
        w[i].Start = ws

    m.addConstr(w.sum() == 1)

    t        = m.addVar(lb=-GRB.INFINITY, name="t")
    var_norm = m.addVar(lb=0, name="var_norm")

    # KEY FIX: express portfolio std via Cholesky factor
    # var_norm = ||L.T w||_2  <=>  var_norm^2 = w^T Sigma_d w
    # Introduce z = L.T w explicitly, then var_norm = ||z||_2
    z = m.addMVar(n, lb=-GRB.INFINITY, name="z")
    m.addConstr(z == L.T @ w)   # linear constraint — Gurobi sees this as affine

    # ||z||_2 <= var_norm  as a standard SOC constraint
    # Gurobi recognizes this form natively as convex
    m.addConstr(var_norm**2 >= z @ z)   # convex quadratic inequality — OK

    # Sharpe epigraph: (mu - P/S)^T w - r >= var_norm * t
    net_mu = mu_d - P / np.array(S)

    if solve_mode == "max_sharpe":
        m.addConstr(net_mu @ w - r >= var_norm * t)
        m.setObjective(t, GRB.MAXIMIZE)
    elif solve_mode == "max_mu":
        m.setObjective(net_mu @ w, GRB.MAXIMIZE)
    elif solve_mode == "min_var":
        m.setObjective(var_norm, GRB.MINIMIZE)
    else:
        raise ValueError("CHOOSE VALID MODE")

    m.optimize()

    weights = np.array([w[i].X for i in range(n)])
    return m, m.ObjVal, weights



In [26]:
def compute_sigma_d(Delta):
    Sigma_d = np.zeros((n, n))

    for i in range(n):
        alpha_i = -1/1.7*np.log(1/Delta[i]-1)
        phi_ai = 1/np.sqrt(2*np.pi)*np.exp(-1/2*alpha_i**2)

        var_i = sigma[i]**2 * (
                (1-Delta[i]) * (1 + alpha_i**2 * Delta[i])
            + alpha_i * phi_ai * (1 - 2*Delta[i])
            - phi_ai * phi_ai)

        Sigma_d[i, i] = var_i

        for j in range(i+1, n):
            rho_ij = Sigma[i, j] / np.sqrt(Sigma[i, i] * Sigma[j, j])

            covar_ij = sigma[i] * sigma[j] * rho_ij*(1-Delta[i])*(1-Delta[j])
            
            Sigma_d[i, j] = covar_ij
            Sigma_d[j, i] = covar_ij

    return Sigma_d

def compute_mu_d(Delta):
    mu_d = np.zeros(n)
    P = np.zeros(n)

    x_pts_CDF = np.linspace(-5, 5, 101)
    y_pts_CDF = -np.array([math.erf(x/np.sqrt(2))*1/2+1/2 for x in x_pts_CDF])

    for i in range(n):
        alpha_i = -1/1.7*np.log(1/Delta[i]-1)
        
        K_expr = (sigma[i]*alpha_i + mu[i] + 1)*S[i]

        d1 = (1 - K_expr/S[i] + T*(r + sigma[i]**2/2))/(sigma[i]*np.sqrt(T))
        d2 = d1 - sigma[i]*np.sqrt(T)

        cdf_d1 = pwl_cdf(d1, x_pts_CDF, y_pts_CDF)
        cdf_d2 = pwl_cdf(d2, x_pts_CDF, y_pts_CDF)
        
        phi_ai = 1/np.sqrt(2*np.pi)*np.exp(-1/2*alpha_i**2)

        P[i] = K_expr*np.exp(-r*T)*cdf_d2 - S[i]*cdf_d1

        mu_d[i] = mu[i] + sigma[i]*(Delta[i]*alpha_i + phi_ai)
    return mu_d - P/S - r

#Delta = np.ones(n)/2
#w = np.array([0.14125627, 0., 0.19738047, 0., 0., 0., 0.08791667, 0., 0., 0.5734466, 0.])
#n_samples = 1000

def mccormick(Delta, w, n_samples, radius=1/100):

    mu_ds = np.zeros(n_samples)
    Sigma_ds = np.zeros(n_samples)

    for k in range(n_samples):
        Delta_tilde = np.clip(Delta + (np.random.uniform()-1/2)*2*radius, 0.1, .5)
        w_tilde = np.clip(w + (np.random.uniform()-1/2)*2*radius, 0.05, .95)
        w_tilde /= np.sum(w_tilde)
        
        mu_d = compute_mu_d(Delta_tilde)
        Sigma_d = compute_sigma_d(Delta_tilde)

        mu_ds[k] = mu_d@w_tilde
        Sigma_ds[k] = np.sqrt(w_tilde.T@Sigma_d@w_tilde)

    mu_max = np.max(mu_ds)
    sigma_max = np.max(Sigma_ds)
    mu_min = np.min(mu_ds)
    sigma_min = np.min(Sigma_ds)

    mu_d = (compute_mu_d(Delta)-r)@w
    Sigma_d = np.sqrt(w.T@compute_sigma_d(Delta)@w)

    UB1 = mu_max/Sigma_d + (mu_d-mu_max)/sigma_max
    UB2 = mu_d/sigma_min + mu_min*(1/Sigma_d-1/sigma_max)
    
    #UB1 = mu_max/Sigma_d+mu_d*mu_min-mu_max*sigma_min
    #UB2 = mu_d*sigma_max+mu_min/Sigma_d-mu_min*sigma_max
    
    return (UB1, UB2)

    print(mu_min, mu_max)
    print(sigma_min, sigma_max)
    print(UB1, UB2)

mccormick(np.ones(n)/2, np.ones(n)/n, 10000, .1)

(np.float64(1.2499336377687904), np.float64(1.3328947657686214))

In [ ]:
import time

time_limit = 100
num_iters = 4
gap = .01/100
mode = "max_sharpe"

num_starting_points = 50
starting_points = []

best_obj = 0
best_model = 0 
w_star = 0
D_star = 0
best_bnd = np.inf
t0 = time.time()

for i in range(num_starting_points):
    D = np.random.random(n)*.4+.1
    w = np.random.random(n)
    w /= np.sum(w)

    starting_points.append((D, w))

header_format = "{:<10} {:<10} {:<10} {:<10} {:<10} {:<10} {:<10}"
row_format = "{:<10} {:<10} {:<10.6f} {:<10.6f} {:<10.6f} {:<10} {:<10}"
data = ["Iteration", "Start. pt.", "Node obj.", "Best obj.", "Best bound", "Gap (%)", "Time (s)"]

print(header_format.format(*data))
for i, (D0, w0) in enumerate(starting_points):
    Delta = D0
    w = w0
    df = pd.DataFrame(index = [i for i in range(num_iters)], columns=["w_obj", "D_obj"])
    
    for k in range(num_iters):
        wm, wobj, w = solve_w_given_delta(Delta, w, 'max_sharpe')
        
        df.at[k, "w_obj"] = wobj

        if wobj > best_obj:
            best_obj = wobj
            w_star = w
            D_star = Delta

        dm, dobj, Delta = solve_delta_given_w(Delta, w)

        df.at[k, "D_obj"] = dobj

        if dobj > best_obj:
            best_obj = dobj
            w_star = w
            D_star = Delta

        df.to_csv(f"sol/dancing_{i}.csv")

        bd = mccormick(Delta, w, 1000, 1/2)

        if best_bnd > bd:
            best_bnd = bd


        data = [i*num_iters + k+1, i+1, max(dobj, wobj), best_obj, best_bnd, f"{round(-(best_obj - best_bnd)/best_obj*100)}%", round(time.time()-t0)]
        print(row_format.format(*data))
        #print(f"Fixed Delta objective: {sigfig(wobj, 5)}, Fixed w objective: {sigfig(dobj, 5)}")


Iteration  Start. pt. Node obj.  Best obj.  Best bound Gap (%)    Time (s)  
1          1          2.169287   2.169287   1.214648   -44%       1         
2          1          2.427171   2.427171   1.214648   -50%       3         
3          1          2.427171   2.427171   1.214648   -50%       4         
4          1          2.427171   2.427171   1.214648   -50%       6         
5          2          2.149244   2.427171   1.214648   -50%       7         

Interrupt request received
6          2          2.422641   2.427171   1.214648   -50%       7         
7          2          2.390746   2.427171   1.214648   -50%       12        
8          2          2.450447   2.450447   1.214648   -50%       13        
9          3          1.778006   2.450447   1.214648   -50%       13        
10         3          2.040577   2.450447   1.214648   -50%       15        
11         3          2.461835   2.461835   1.214648   -51%       16        
12         3          2.461835   2.461835   1.21

In [65]:
wm.ObjVal

2.4231922538485406

In [33]:
var = wm.getVars()

weights = [float(v.X) for v in var if "weight" in v.VarName]
deltas = [float(v.X) for v in var if "Delta" in v.VarName]

# REMOVING WEIGHTS BELOW 0.1%
w = np.array([w if w > 0.001 else 0 for w in weights ])
w /= w.sum()
print(deltas)
print("""
\\begin{table}[!h]
    \\centering
    \\begin{tabular}{|c|c|c|}""")

print("\\hline Ticker & Weight & Delta \\\\ \\hline")

for i, ticker in enumerate(all_stocks):
    if w[i]:
        print(f"{ticker} &  {round(w[i]*100, 2)}\\% & {round(deltas[i], 2)} \\\\ \\hline")
print("""
    \\end{tabular}
\\end{table}
""")

[]

\begin{table}[!h]
    \centering
    \begin{tabular}{|c|c|c|}
\hline Ticker & Weight & Delta \\ \hline


IndexError: list index out of range

140000.0